In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/PCB_DATASET.zip" -d "/content/pcb_data"

In [ ]:
!pip install python-docx

In [ ]:
import docx
import os

def find_file(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)
    return None

def read_docx(file_path):
    if not os.path.exists(file_path):
        return f"Error: File {file_path} not found."
    doc = docx.Document(file_path)
    full_text = []
    for para in doc.paragraphs:
        full_text.append(para.text)
    return '\n'.join(full_text)

# Broad search in both /content and /content/drive
target_filename = 'A3_CV(8A).docx'
file_path = find_file(target_filename, '/content')

if not file_path:
    file_path = find_file(target_filename, '/content/drive')

if file_path:
    print(f"Found file at: {file_path}")
    assignment_text = read_docx(file_path)
    print("--- Assignment Document Content ---")
    print(assignment_text)
else:
    print(f"File '{target_filename}' not found. Let's list the extracted pcb_data contents to see if we can find relevant info there.")
    !ls -R /content/pcb_data | head -n 20

In [ ]:
import os

# Define paths
image_root = '/content/pcb_data/images'
anno_root = '/content/pcb_data/Annotations'

# 1. How many images
all_images = []
for root, dirs, files in os.walk(image_root):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append(file)

print(f'Total number of images: {len(all_images)}')

# 2. Attributes and Class Imbalance
classes = [d for d in os.listdir(anno_root) if os.path.isdir(os.path.join(anno_root, d))]
print(f'Attributes (Classes) identified: {classes}')
print(f'Number of attributes: {len(classes)}')

print('\n--- Class Distribution ---')
class_counts = {}
for cls in classes:
    cls_path = os.path.join(anno_root, cls)
    num_annos = len([f for f in os.listdir(cls_path) if f.endswith('.xml')])
    class_counts[cls] = num_annos
    print(f'{cls}: {num_annos} annotations')

# Assess imbalance
max_val = max(class_counts.values())
min_val = min(class_counts.values())
is_imbalanced = 'Yes' if max_val > 1.5 * min_val else 'No'

print(f'\nIs the class imbalanced? {is_imbalanced}')

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Define Transformations
# Training: Augmentation + Normalization
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test: Just Resize + Normalization
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Load the full dataset (using ImageFolder assumes classes are in subfolders)
full_dataset = datasets.ImageFolder(root='/content/pcb_data/images', transform=train_transform)

# 3. Stratified Split: 70% Train, 15% Val, 15% Test
targets = full_dataset.targets
train_idx, temp_idx = train_test_split(np.arange(len(targets)), test_size=0.3, random_state=42, stratify=targets)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=[targets[i] for i in temp_idx])

# Create Subsets
train_dataset = Subset(full_dataset, train_idx)
# Override transform for val/test subsets
val_dataset = Subset(datasets.ImageFolder(root='/content/pcb_data/images', transform=val_test_transform), val_idx)
test_dataset = Subset(datasets.ImageFolder(root='/content/pcb_data/images', transform=val_test_transform), test_idx)

# 4. Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Dataset Split Complete:")
print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchsummary import summary

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(SimpleCNN, self).__init__()
        # Layer 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        # Layer 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        # Layer 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        # Layer 4
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        # Layer 5
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        # After 5 poolings on 224x224: 224->112->56->28->14->7
        self.fc1 = nn.Linear(512 * 7 * 7, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.pool(F.relu(self.bn5(self.conv5(x))))

        x = x.view(-1, 512 * 7 * 7)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes=6).to(device)

# Print Summary
summary(model, (3, 224, 224))

In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt

# 1. Hyperparameters
num_epochs = 10
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 2. Tracking metrics
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

# 3. Training Loop
for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()

    # Validation
    model.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()

    # Calculate averages
    train_loss /= len(train_loader.dataset)
    train_acc = train_correct / len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)
    val_acc = val_correct / len(val_loader.dataset)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

# 4. Save Model
torch.save(model.state_dict(), 'simple_cnn_pcb.pth')
print("Model saved as simple_cnn_pcb.pth")

In [ ]:
import matplotlib.pyplot as plt

# Plotting training & validation accuracy
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plotting training & validation loss
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
test_correct = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        test_correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = test_correct / len(test_loader.dataset)
print(f'Final Test Accuracy: {test_acc:.4f}')

In [ ]:
import torchvision.models as models
import torch.optim as optim
import torch.nn as nn

# 1. Load Pre-trained ResNet18
resnet_model = models.resnet18(weights='DEFAULT')

# 2. Modify the final fully connected layer for 6 classes
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, 6)
resnet_model = resnet_model.to(device)

# 3. Hyperparameters and Optimizer
num_epochs_resnet = 10
optimizer_resnet = optim.Adam(resnet_model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

resnet_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

# 4. ResNet Training Loop
print("Starting ResNet18 Training...")
for epoch in range(num_epochs_resnet):
    resnet_model.train()
    train_loss, train_correct = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_resnet.zero_grad()
        outputs = resnet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_resnet.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()

    resnet_model.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()

    resnet_history['train_loss'].append(train_loss/len(train_loader.dataset))
    resnet_history['val_loss'].append(val_loss/len(val_loader.dataset))
    resnet_history['train_acc'].append(train_correct/len(train_loader.dataset))
    resnet_history['val_acc'].append(val_correct/len(val_loader.dataset))

    print(f"ResNet Epoch [{epoch+1}/{num_epochs_resnet}] - Train Acc: {resnet_history['train_acc'][-1]:.4f}, Val Acc: {resnet_history['val_acc'][-1]:.4f}")

torch.save(resnet_model.state_dict(), 'resnet_pcb.pth')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def evaluate_model(model, loader, title):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"\n--- {title} Classification Report ---")
    # Map numerical labels back to class names
    class_names = train_dataset.dataset.classes
    print(classification_report(all_labels, all_preds, target_names=class_names))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.title(f'Confusion Matrix: {title}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

# Evaluate ResNet on test set
evaluate_model(resnet_model, test_loader, "ResNet18")

In [ ]:
import xml.etree.ElementTree as ET
import cv2
from google.colab.patches import cv2_imshow
import os

def visualize_pcb_sample(img_path, xml_path):
    image = cv2.imread(img_path)
    if image is None:
        print(f"Could not read image at {img_path}")
        return
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for obj in root.findall('object'):
        label = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        cv2.putText(image, label, (xmin, ymin-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)
    cv2_imshow(image)

# Visualize one sample from 'Missing_hole'
sample_img = '/content/pcb_data/images/Missing_hole/01_missing_hole_01.jpg'
sample_xml = '/content/pcb_data/Annotations/Missing_hole/01_missing_hole_01.xml'

if os.path.exists(sample_img) and os.path.exists(sample_xml):
    print("Visualizing sample with bounding boxes to understand data:")
    visualize_pcb_sample(sample_img, sample_xml)
else:
    print("Sample files not found for visualization check.")

In [ ]:
!pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np

# Targeted layer for ResNet18 (last convolutional layer)
target_layers = [resnet_model.layer4[-1]]
cam = GradCAM(model=resnet_model, target_layers=target_layers)

# Get a sample image from the test loader
input_tensor, labels = next(iter(test_loader))
input_tensor = input_tensor[0:1].to(device)

grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(labels[0].item())])
grayscale_cam = grayscale_cam[0, :]

# Denormalize image for visualization
img_show = input_tensor[0].cpu().permute(1, 2, 0).numpy()
img_show = (img_show * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
img_show = np.clip(img_show, 0, 1)

visualization = show_cam_on_image(img_show, grayscale_cam, use_rgb=True)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(img_show)
plt.subplot(1, 2, 2)
plt.title("Grad-CAM Heatmap")
plt.imshow(visualization)
plt.show()

In [ ]:
import xml.etree.ElementTree as ET
import cv2
from google.colab.patches import cv2_imshow
import os

def visualize_pcb_sample(img_path, xml_path):
    image = cv2.imread(img_path)
    if image is None:
        print(f"Could not read image at {img_path}")
        return
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for obj in root.findall('object'):
        label = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        cv2.putText(image, label, (xmin, ymin-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)
    cv2_imshow(image)

# Visualize one sample from 'Missing_hole'
sample_img = '/content/pcb_data/images/Missing_hole/01_missing_hole_01.jpg'
sample_xml = '/content/pcb_data/Annotations/Missing_hole/01_missing_hole_01.xml'

if os.path.exists(sample_img) and os.path.exists(sample_xml):
    print("Visualizing sample with bounding boxes to check data characteristics:")
    visualize_pcb_sample(sample_img, sample_xml)
else:
    print("Sample files not found for visualization check.")

In [ ]:
!pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np

# Targeted layer for ResNet18 (usually the last convolutional layer)
target_layers = [resnet_model.layer4[-1]]
cam = GradCAM(model=resnet_model, target_layers=target_layers)

# Get a sample image from the test loader
input_tensor, labels = next(iter(test_loader))
input_tensor = input_tensor[0:1].to(device)

grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(labels[0].item())])
grayscale_cam = grayscale_cam[0, :]

# Denormalize image for visualization
img_show = input_tensor[0].cpu().permute(1, 2, 0).numpy()
img_show = (img_show * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
img_show = np.clip(img_show, 0, 1)

visualization = show_cam_on_image(img_show, grayscale_cam, use_rgb=True)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(img_show)
plt.subplot(1, 2, 2)
plt.title("Grad-CAM Heatmap")
plt.imshow(visualization)
plt.show()

In [ ]:
import xml.etree.ElementTree as ET
import cv2
from google.colab.patches import cv2_imshow
import os

def visualize_pcb_sample(img_path, xml_path):
    image = cv2.imread(img_path)
    if image is None:
        print(f"Could not read image at {img_path}")
        return
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for obj in root.findall('object'):
        label = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        cv2.putText(image, label, (xmin, ymin-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)
    cv2_imshow(image)

# Visualize one sample from 'Missing_hole'
sample_img = '/content/pcb_data/images/Missing_hole/01_missing_hole_01.jpg'
sample_xml = '/content/pcb_data/Annotations/Missing_hole/01_missing_hole_01.xml'

if os.path.exists(sample_img) and os.path.exists(sample_xml):
    print("Visualizing sample with bounding boxes:")
    visualize_pcb_sample(sample_img, sample_xml)
else:
    print("Sample files not found for visualization check.")

In [ ]:
!pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np

# Targeted layer for ResNet18
target_layers = [resnet_model.layer4[-1]]
cam = GradCAM(model=resnet_model, target_layers=target_layers)

# Get a sample image from the test loader
input_tensor, labels = next(iter(test_loader))
input_tensor = input_tensor[0:1].to(device)

grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(labels[0].item())])
grayscale_cam = grayscale_cam[0, :]

# Denormalize image for visualization
img_show = input_tensor[0].cpu().permute(1, 2, 0).numpy()
img_show = (img_show * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
img_show = np.clip(img_show, 0, 1)

visualization = show_cam_on_image(img_show, grayscale_cam, use_rgb=True)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(img_show)
plt.subplot(1, 2, 2)
plt.title("Grad-CAM Heatmap")
plt.imshow(visualization)
plt.show()

In [ ]:
import xml.etree.ElementTree as ET
import cv2
from google.colab.patches import cv2_imshow

def visualize_pcb_sample(img_path, xml_path):
    image = cv2.imread(img_path)
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for obj in root.findall('object'):
        label = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        cv2.putText(image, label, (xmin, ymin-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)
    cv2_imshow(image)

# Visualize one sample from 'Missing_hole'
sample_img = '/content/pcb_data/images/Missing_hole/01_missing_hole_01.jpg'
sample_xml = '/content/pcb_data/Annotations/Missing_hole/01_missing_hole_01.xml'
if os.path.exists(sample_img) and os.path.exists(sample_xml):
    visualize_pcb_sample(sample_img, sample_xml)
else:
    print("Sample files not found for visualization check.")

In [ ]:
!pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Targeted layer for ResNet18
target_layers = [resnet_model.layer4[-1]]
cam = GradCAM(model=resnet_model, target_layers=target_layers)

# Get a sample image from the test loader
input_tensor, labels = next(iter(test_loader))
input_tensor = input_tensor[0:1].to(device)

grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(labels[0].item())])
grayscale_cam = grayscale_cam[0, :]

# Denormalize image for visualization
img_show = input_tensor[0].cpu().permute(1, 2, 0).numpy()
img_show = (img_show * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
img_show = np.clip(img_show, 0, 1)

visualization = show_cam_on_image(img_show, grayscale_cam, use_rgb=True)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(img_show)
plt.subplot(1, 2, 2)
plt.title("Grad-CAM Heatmap")
plt.imshow(visualization)
plt.show()

In [ ]:
import torchvision.models as models

# 1. Load Pre-trained ResNet18
resnet_model = models.resnet18(pretrained=True)

# 2. Modify the final fully connected layer for our 6 classes
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, 6)

resnet_model = resnet_model.to(device)

# 3. Hyperparameters and Optimizer
num_epochs_resnet = 10
optimizer_resnet = optim.Adam(resnet_model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

resnet_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

# 4. ResNet Training Loop
for epoch in range(num_epochs_resnet):
    resnet_model.train()
    train_loss, train_correct = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_resnet.zero_grad()
        outputs = resnet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_resnet.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()

    resnet_model.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()

    resnet_history['train_loss'].append(train_loss/len(train_loader.dataset))
    resnet_history['val_loss'].append(val_loss/len(val_loader.dataset))
    resnet_history['train_acc'].append(train_correct/len(train_loader.dataset))
    resnet_history['val_acc'].append(val_correct/len(val_loader.dataset))

    print(f"ResNet Epoch [{epoch+1}/{num_epochs_resnet}] - Val Acc: {resnet_history['val_acc'][-1]:.4f}")

torch.save(resnet_model.state_dict(), 'resnet_pcb.pth')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

def evaluate_model(model, loader, title):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"\n--- {title} Classification Report ---")
    print(classification_report(all_labels, all_preds, target_names=classes))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues')
    plt.title(f'Confusion Matrix: {title}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

# Evaluate ResNet
evaluate_model(resnet_model, test_loader, "ResNet18")

After reading the document content, I will analyze the text to answer your questions regarding the attributes, class imbalance, and the total number of images.